# ECA-0.5 — Parallel Byte Streaming + Matrix Training (T4)

This experiment keeps raw UTF-8 bytes (0–255) plus EOS=256. A background producer incrementally fetches and encodes documents into bounded per-lane queues. The GPU consumes a batch of independent lanes together; each lane keeps its own recurrent state across chunks. No corpus-wide document list, BPE/SuperBPE, attention, Transformer blocks, or positional embeddings.

In [ ]:
import os, sys, torch, platform
print('Python:', platform.python_version(), '| PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'Select Runtime → Change runtime type → T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
!nvidia-smi

In [ ]:
REPO='/content/eca'
BRANCH='eca-0.5-parallel-streaming'
if not os.path.isdir(REPO):
    !git clone --branch $BRANCH https://github.com/swasthik-s/eca.git $REPO
%cd /content/eca
!git fetch origin
!git checkout $BRANCH
!git pull --ff-only origin $BRANCH
!pip -q install -e '.[train,dev]'
print('Checked out:', BRANCH)

In [ ]:
!pytest -q tests/test_parallel_matrix.py tests/test_parallel_stream.py tests/test_adaptive_recovery.py
print('Parallel matrix, stream and recovery tests passed.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE='/content/drive/MyDrive/ECA/checkpoints'
os.makedirs(BASE, exist_ok=True)
print('Persistent checkpoint directory:', BASE)

## T4 experiment configuration

`DOCUMENTS` is an upper bound on source documents, not a preload count. The dataset iterator remains lazy. Validation is taken from the initial documents and held separately; training documents are streamed into bounded lane queues. Increase the document cap without increasing RAM usage proportionally.

In [ ]:
DATASET_ID='HuggingFaceFW/fineweb-edu'
DATASET_CONFIG='sample-10BT'
DOCUMENTS=100000
VALIDATION_BYTES=200000
BUFFER_SYMBOLS=262144  # total target; divided across batch lanes
LOW_WATERMARK=0.30
HIGH_WATERMARK=0.90
STEPS=1500
SEQUENCE_LENGTH=256
BATCH_SIZE=8
HIDDEN_SIZE=256
EMBEDDING_SIZE=64
DEPTH=2
KERNEL_SIZE=5
SEED=17
EVAL_EVERY=50
PATIENCE=3
RECOVERY_EXTRA_STEPS=200
MAX_RECOVERIES=3
RUN_NAME=f'eca_0_5_h{HIDDEN_SIZE}_d{DEPTH}_b{BATCH_SIZE}'
BEST=f'{BASE}/{RUN_NAME}_best.pt'
LATEST=f'{BASE}/{RUN_NAME}_latest.pt'
print('Run:', RUN_NAME, '| initial steps:', STEPS, '| max recovery steps:', RECOVERY_EXTRA_STEPS*MAX_RECOVERIES)

## Launch training

The producer runs asynchronously while the GPU trains on batches. Each lane receives whole documents round-robin and preserves its own order/state. Low/high watermarks provide backpressure. The model still processes each lane causally; parallelism is across independent lanes and across the ECA associative scan, not by allowing future bytes to affect earlier predictions.

In [ ]:
import subprocess
cmd = [sys.executable, '-m', 'eca.training.train_parallel_stream',
 '--dataset','hf','--dataset-id',DATASET_ID,'--dataset-config',DATASET_CONFIG,
 '--documents',str(DOCUMENTS),'--validation-bytes',str(VALIDATION_BYTES),
 '--buffer-symbols',str(BUFFER_SYMBOLS),'--low-watermark',str(LOW_WATERMARK),
 '--high-watermark',str(HIGH_WATERMARK),'--steps',str(STEPS),
 '--sequence-length',str(SEQUENCE_LENGTH),'--batch-size',str(BATCH_SIZE),
 '--hidden-size',str(HIDDEN_SIZE),'--embedding-size',str(EMBEDDING_SIZE),
 '--depth',str(DEPTH),'--kernel-size',str(KERNEL_SIZE),'--seed',str(SEED),
 '--eval-every',str(EVAL_EVERY),'--patience',str(PATIENCE),
 '--recovery-extra-steps',str(RECOVERY_EXTRA_STEPS),'--max-recoveries',str(MAX_RECOVERIES),
 '--output',LATEST,'--best-checkpoint',BEST]
print('Launching:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import os, torch
assert os.path.isfile(BEST), f'Best checkpoint missing: {BEST}'
best=torch.load(BEST, map_location='cpu', weights_only=True)
print('Architecture:', best['architecture'])
print('Best step:', best['step'])
print('Best validation BPB:', best['validation_bpb'])
print('Model config:', best['config'])
print('Best checkpoint MB:', round(os.path.getsize(BEST)/(1024**2),2))
print('Latest checkpoint exists:', os.path.isfile(LATEST))
!nvidia-smi

## Interpreting the run

- `buffered=` reports queued byte symbols across lanes; it should remain bounded by the configured per-lane capacities.
- `bytes_per_sec` measures training input throughput, not network download throughput.
- `valid_bpb` is held-out next-byte prediction. Lower is better, but it does not prove reasoning or semantic understanding.
- A recovery restores the best model/optimizer/scaler, lowers learning rate and increases the step budget within the configured limit.
- This version uses one asynchronous dataset producer and parallel GPU lanes. It does **not** yet use multiple independent network-download workers; that is a separate optimization to benchmark after correctness and GPU utilization are measured.

For a first run, keep these settings. If the GPU is underutilized, profile `nvidia-smi` and increase batch size or sequence length one variable at a time.